### General information

This python module uses the following external packages:
- pandas
- pyam-iamc (https://github.com/IAMconsortium/pyam)
- nomenclature-iamc (https://github.com/IAMconsortium/nomenclature)
- gdxpds (https://github.com/NREL/gdx-pandas) with gdxcc
- GitPython (for the validation, see below)

All of them are installed with `conda env create -f environment.yml` in the folder `iamc_conversion` (then `conda activate genesysmod-iamc`). The validation needs Python 3.11 or newer.

#### File structure of the module
- __this_file__ : Jupyter notebook with general guidelines
- /genesysmod_to_iamc : Folder with source code and files
 - /genesysmod_to_iamc/mappings/ : Folder with mappings of GENeSYS-MOD variables to IAMC variables
 - /genesysmod_to_iamc/input/ : Folder with original GENeSYS-MOD pathway results
 - /genesysmod_to_iamc/out/ : Folder with generated csv files in IAMC format
 - /genesysmod_to_iamc/out/combined_excel/ : Folder with combined excel files from generated csv files

**Validation:** The combined Excel file is checked automatically with the same workflow that the scenario explorer runs on upload. The project's workflow repository is set in *genesysmod_to_iamc/\_statics.py* (*DEF_WORKFLOW_REPO_URL*, currently https://github.com/iiasa/idesignres-workflow; for another project, change it to that project's repository). The repository is downloaded to the folder *.genesysmod_to_iamc* in your user folder and updated before every check, together with the IAMC common-definitions, so no definitions need to be downloaded by hand. The result is written to *out/validation_report.txt*.

#### Changes to the scripts required to run the conversion
This script uses gdxpds which was unable to succesfully locate the GAMS directory by itself in its newest version. Therefore, the path to GAMS needs to be set manually. **You will need to change the directory such that it correctly points to your installed GAMS distribution.** The variable is called *DEF_GAMS_DIR* and is set in the file *genesysmod_to_iamc/\_statics.py*.

Also, you can change the read-in filenames, model name, etc. in the same file.

### GENeSYS-MOD to openEntrance converter

This jupyter notebook shows an exemplary conversion of GENeSYS-MOD results to the common openENTRANCE data fromat. The outputs of the scripts in this module will be csv files, that represent GENeSYS-MOD data in openENTRANCE nomenclature. First, the corresponding scripts will be imported as module to this notebook:

In [ ]:
import genesysmod_to_iamc

Next, the actual result file conversion can be started by calling ```genesysmod_to_iamc.generate_data(FileName: str, FileType: str)```. The filename can be passed as the first argument in the function call as a string, without the file extension. As a second argument, the file type has to be specified. Currently, acceptable file types are ``'gdx'`` and ``'csv'``.

When using the `'csv'` file type, the output of GENeSYS-MOD's Julia version is expected. This output consists of several csv files, which names start with the corresponding output type, followed by the name of the specific model run. E.g. `'output_energy_balance_Sample_Model_Run.csv'`. It's important that the files keep that naming convention. If the `'csv'` file type is used, the name of the model run (the second part of the output file name) is used as the file name for the first argument in the function call.

For the ease of use, the module ```genesysmod_to_iamc``` also includes an enumeration ```Pathways``` that includes references to the filenames for the openEntrance Pathways. Cells for the openEntrance output can be found at the end of the notebook:

```
class Pathways(Enum):
    TF = "TechnoFriendly"
    DT = "DirectedTransition"
    GD = "GradualDevelopment"
    SC = "SocietalCommitment"
```

In [ ]:
generate_series_data = False
generate_load_factors = False
generate_transmission_data = False
combine_outputs = True

Furthermore, ```genesysmod_to_iamc.generate_data(FileName : str)``` has four optional parameters ```generate_series_data```, ```generate_load_factors```, ```generate_transmission_data```, and ```combine_outputs``` that all default to ```False```. 

If ```generate_series_data``` is set to ```True```, full hourly demand time series for Final Energy|Residential and Commercial|Electricity, Final Energy|Industry|Electricity, Final Energy|Transportation|Electricity, Final Energy|Electricity, Final Energy|Residential and Commercial and Final Energy|Industry will be generated. Full hourly demand series will be written to an extra file. **Note:** setting this parameter will significantly increase the computing time.

If ```generate_load_factors``` is set to ```True```, full hourly load series for Solar PV, Wind Onshore and Wind Offshore will be generated. Full hourly load series will be written to an extra file. **Note:** setting this parameter will significantly increase the computing time.

**Note:** For both the series data and load factors, an input excel file in the GENeSYS-MOD format is expected. The file name needs to start with "Timeseries_" followed by the gdx file name or the name of the model run for gdx or csv file type respectively (E.g: "Timeseries_Sample_Model_Run").

If ```generate_transmission_data``` is set to true, the electricity grid information like trade flows and net trade capacities will be output as well.

If ```combine_outputs``` is set to ```True```, the different result files for a pathway (yearly, hourly, load factors) can be combined to one single file.
_____

Example creation of results for a gdx input file:

In [ ]:
genesysmod_to_iamc.generate_data("Sample_Model_Run",
                                 "gdx",
                                 generate_series_data, 
                                 generate_load_factors,
                                 generate_transmission_data,
                                 combine_outputs)

Example creation of results for csv input files:

In [ ]:
genesysmod_to_iamc.generate_data("Sample_Model_Run",
                                 "csv",
                                 generate_series_data, 
                                 generate_load_factors,
                                 generate_transmission_data,
                                 combine_outputs)

Results will be written to `genesysmod_to_iamc/out`. The following line of code creates a unified Excel file with all pathways combined (only yearly values are aggregated due to the file size limit of Excel files) and validates it (see above). An existing file can be checked on its own with `genesysmod_to_iamc.validate("path/to/file.xlsx")`.

In [ ]:
genesysmod_to_iamc.generate_combined_excel_yearly()

______
Creation of openEntrance pathways:

In [ ]:
genesysmod_to_iamc.generate_data(genesysmod_to_iamc.Pathways.TF.value,
                                 "gdx",
                                 generate_series_data, 
                                 generate_load_factors,
                                 generate_transmission_data,
                                 combine_outputs)

genesysmod_to_iamc.generate_data(genesysmod_to_iamc.Pathways.SC.value,
                                 "gdx",
                                 generate_series_data, 
                                 generate_load_factors, 
                                 generate_transmission_data,
                                 combine_outputs)

genesysmod_to_iamc.generate_data(genesysmod_to_iamc.Pathways.GD.value,
                                 "gdx",
                                 generate_series_data, 
                                 generate_load_factors, 
                                 generate_transmission_data,
                                 combine_outputs)

genesysmod_to_iamc.generate_data(genesysmod_to_iamc.Pathways.DT.value, 
                                 "gdx",
                                 generate_series_data, 
                                 generate_load_factors, 
                                 generate_transmission_data,
                                 combine_outputs)

genesysmod_to_iamc.generate_combined_excel_yearly()